In [1]:
import polars as pl
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
drive_path = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'

In [4]:
split_dir = f"{drive_path}/processed_data/final_foundation/splits"

In [5]:
# Load data
train = pl.read_parquet(f"{split_dir}/train.parquet")
test = pl.read_parquet(f"{split_dir}/test.parquet")

In [8]:
train.describe()

statistic,Username,BGGId,Rating,user_id,item_id,split
str,str,f64,f64,f64,f64,str
"""count""","""14929223""",1.4929223e7,1.4929223e7,1.4929223e7,1.4929223e7,"""14929223"""
"""null_count""","""0""",0.0,0.0,0.0,0.0,"""0"""
"""mean""",null,109751.589693,7.11754,212228.569534,10445.06234,null
"""std""",null,92745.009977,1.53717,118617.743476,5990.115588,null
"""min""",""" Fu_Koios """,1.0,0.0001,0.0,0.0,"""train"""
"""25%""",null,15987.0,6.0,109829.0,6062.0,null
"""50%""",null,107529.0,7.0,218176.0,10998.0,null
"""75%""",null,181279.0,8.0,314292.0,14989.0,null
"""max""","""Æleksandr Þræð""",349161.0,10.0,411374.0,21924.0,"""train"""


In [9]:
test.describe()

statistic,Username,BGGId,Rating,user_id,item_id,split
str,str,f64,f64,f64,f64,str
"""count""","""2124050""",2.12405e6,2.12405e6,2.12405e6,2.12405e6,"""2124050"""
"""null_count""","""0""",0.0,0.0,0.0,0.0,"""0"""
"""mean""",null,111777.453736,7.240908,211351.051697,10564.613659,null
"""std""",null,93314.797546,1.586519,118659.380427,6010.668202,null
"""min""",""" Fu_Koios """,1.0,0.5,0.0,0.0,"""test"""
"""25%""",null,16992.0,6.0,108786.0,6188.0,null
"""50%""",null,112686.0,7.0,216518.0,11146.0,null
"""75%""",null,182874.0,8.0,313466.0,15118.0,null
"""max""","""Æleksandr Þræð""",349161.0,10.0,411374.0,21924.0,"""test"""


In [10]:
train.columns

['Username', 'BGGId', 'Rating', 'user_id', 'item_id', 'split']

In [11]:
train.head()

Username,BGGId,Rating,user_id,item_id,split
str,i64,f64,u32,u32,str
"""GameTutor""",172,9.0,69172,151,"""train"""
"""Arkhamhoarder""",120677,8.0,11860,11384,"""train"""
"""Bjoern23""",191177,8.0,21564,15523,"""train"""
"""DoveBar""",1111,6.0,51369,849,"""train"""
"""titillo80""",157403,6.0,391574,13412,"""train"""


In [6]:
# 1. Calculate Global Mean (C) and Minimum Ratings required (m)
C = train["Rating"].mean()
m = 50 # Minimum ratings to be considered "popular"


In [12]:
# 2. Calculate Weighted Rating for each game
# Formula: (v/(v+m) * R) + (m/(v+m) * C)
stats = train.group_by("item_id").agg([
    pl.len().alias("v"),
    pl.col("Rating").mean().alias("R")
])

# C and m are Python scalars, so we use them directly in the expression
popularity_model = stats.with_columns(
    ((pl.col("v") / (pl.col("v") + m)) * pl.col("R") +
     (m / (pl.col("v") + m)) * C).alias("pop_score")
)

In [14]:
print(C)

7.11753981821462


In [13]:
# 3. Evaluate: Join with test set and calculate RMSE
test_results = test.join(popularity_model, on="item_id", how="left").fill_null(C)
rmse_pop = np.sqrt(((test_results["Rating"] - test_results["pop_score"])**2).mean())

print(f"Popularity Baseline RMSE: {rmse_pop:.4f}")

Popularity Baseline RMSE: 1.3818
